# EPIC Clarity Visit Detail Hydration

This notebook hydrates the OMOP VISIT_DETAIL table from EPIC Clarity ADT (Admit/Discharge/Transfer) events.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_CLARITY_ADT` - Admission/discharge/transfer events
- `_exponent._bronze_epic_clarity_*.dbo_PAT_ENC_HSP` - Hospital encounter with patient class information

## OMOP Fields Populated
- visit_detail_id (surrogate key)
- visit_detail_source_value
- visit_detail_concept_id (mapped from event type/patient class)
- visit_detail_start_datetime (ADT effective time)
- visit_occurrence_id (parent encounter)
- care_site_id (bed/unit location if available)

## Notes
- VISIT_DETAIL represents intra-hospital movements
- Captures ADT events (admission, discharge, transfer)
- Links to parent visit_occurrence for the overall hospital stay
- Useful for tracking patient movement across units/beds

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC ADT and patient class data for visit details
%sql
CREATE OR REPLACE TEMP VIEW visit_detail_silver AS
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_ADT', 'EVENT_ID', ca.EVENT_ID) AS visit_detail_source_value,
    ca.EVENT_TYPE_C_NAME AS visit_detail_type_source_value,
    ca.EFFECTIVE_TIME AS visit_detail_start_datetime,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', ca.PAT_ENC_CSN_ID) AS visit_occurrence_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'BED_ID', 'BED_ID', ca.BED_ID_BED_LABEL) AS care_site_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_CLARITY_ADT ca
WHERE ca.EVENT_ID IS NOT NULL
UNION ALL
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC_HSP', 'PAT_ENC_CSN_ID', peh.PAT_ENC_CSN_ID) AS visit_detail_source_value,
    peh.ADT_PAT_CLASS_C_NAME AS visit_detail_type_source_value,
    peh.CONTACT_DATE AS visit_detail_start_datetime,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', peh.PAT_ENC_CSN_ID) AS visit_occurrence_source_value,
    NULL AS care_site_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_PAT_ENC_HSP peh
WHERE peh.PAT_ENC_CSN_ID IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.visit_detail AS target
USING visit_detail_silver AS source
ON target.visit_detail_source_value = source.visit_detail_source_value

WHEN MATCHED AND NOT (
    target.visit_detail_type_source_value <=> source.visit_detail_type_source_value
    AND target.visit_detail_start_datetime <=> source.visit_detail_start_datetime
    AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
    AND target.care_site_source_value <=> source.care_site_source_value
)
THEN UPDATE SET
    target.visit_detail_type_source_value = source.visit_detail_type_source_value,
    target.visit_detail_start_datetime = source.visit_detail_start_datetime,
    target.visit_occurrence_source_value = source.visit_occurrence_source_value,
    target.care_site_source_value = source.care_site_source_value,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    visit_detail_source_value,
    visit_detail_type_source_value,
    visit_detail_start_datetime,
    visit_occurrence_source_value,
    care_site_source_value,
    updated_tsp
)
VALUES (
    source.visit_detail_source_value,
    source.visit_detail_type_source_value,
    source.visit_detail_start_datetime,
    source.visit_occurrence_source_value,
    source.care_site_source_value,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_detail (
    source_system,
    visit_detail_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.visit_detail_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT visit_detail_source_value, updated_tsp
    FROM _exponent.omop_silver.visit_detail
    WHERE visit_detail_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visit_detail x
    ON s.visit_detail_source_value = x.visit_detail_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with visit_occurrence, care_site, and concept mappings
%sql
CREATE OR REPLACE TEMP VIEW visit_detail_gold AS
SELECT
    m.visit_detail_id,
    mv.visit_occurrence_id,
    COALESCE(vcm.concept_id, 0) AS visit_detail_concept_id,
    s.visit_detail_start_datetime,
    mc.care_site_id,
    s.updated_tsp
FROM _exponent.omop_silver.visit_detail s
INNER JOIN _exponent.omop_mapping.source_to_visit_detail m
    ON s.visit_detail_source_value = m.visit_detail_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence mv
    ON s.visit_occurrence_source_value = mv.visit_occurrence_source_value
    AND mv.source_system = 'epic_clarity'
    AND mv.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept vcm
    ON vcm.source_id = s.visit_detail_type_source_value
    AND vcm.domain_id = 'Visit'
    AND vcm.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.source_to_care_site mc
    ON s.care_site_source_value = mc.care_site_source_value
    AND mc.source_system = 'epic_clarity'
    AND mc.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.visit_detail AS target
USING visit_detail_gold AS source
ON target.visit_detail_id = source.visit_detail_id

WHEN MATCHED AND NOT (
    target.visit_occurrence_id <=> source.visit_occurrence_id
    AND target.visit_detail_concept_id <=> source.visit_detail_concept_id
    AND target.visit_detail_start_datetime <=> source.visit_detail_start_datetime
    AND target.care_site_id <=> source.care_site_id
)
THEN UPDATE SET
    target.visit_occurrence_id = source.visit_occurrence_id,
    target.visit_detail_concept_id = source.visit_detail_concept_id,
    target.visit_detail_start_datetime = source.visit_detail_start_datetime,
    target.care_site_id = source.care_site_id

WHEN NOT MATCHED THEN INSERT (
    visit_detail_id,
    visit_occurrence_id,
    visit_detail_concept_id,
    visit_detail_start_datetime,
    care_site_id
)
VALUES (
    source.visit_detail_id,
    source.visit_occurrence_id,
    source.visit_detail_concept_id,
    source.visit_detail_start_datetime,
    source.care_site_id
)